In [1]:
import torch
print("CUDA :", torch.cuda.is_available())  # doit afficher True

CUDA : True


In [2]:
!nvidia-smi   # doit montrer une Tesla T4

Sat Sep 12 18:34:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -U langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain_chroma
!pip install torch
!pip install -U "bitsandbytes>=0.46.1" transformers accelerate
!pip install rouge-score
!pip install pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [4]:
import pandas as pd
dataset = pd.read_excel('ohada.xlsx')

In [6]:
dataset.head()

,title,content,details
0,"| OHADA, Cour commune de justice et d'arbitrag...",COUR COMMUNE DE JUSTICE ET ARBITRAGEDemande d'...,Analyses\nVOIES D'EXECUTION - ACTES DE PROCE...
1,"| OHADA, Cour commune de justice et d'arbitrag...",COUR COMMUNE DE JUSTICE ET ARBITRAGEDemande d’...,Analyses\nVOIES D'EXÉCUTION - ACTES DE PROCÉDU...
2,"| OHADA, Cour commune de justice et d'arbitrag...",ORGANISATION POUR L’HARMONISATION EN AFRI...,Références :\nOhada.com/Unida\nOrigine de la d...
3,"| OHADA, Cour commune de justice et d'arbitrag...",ORGANISATION POUR L’HARMONISATION EN AFRIQU...,Analyses\nEXÉCUTION PROVISOIRE - DÉFENSES À EX...
4,"| OHADA, Cour commune de justice et d'arbitrag...",ORGANISATION POUR L'HARMONISATION EN AFRIQUE D...,Analyses\nPOURVOI EN CASSATION - DEFAUT DE P...


In [9]:
from langchain_core.documents import Document
import pandas as pd

documents = []

for _, row in dataset.iterrows():
    content = f"""
    title : {row['title']}
    content : {row['content']}
    details : {row['details']}
    """

    documents.append(
        Document(
            page_content=content,
            metadata={
                "title": row['title'],
                "content": row['content'],
                "details": row['details']
            }
        )
    )

In [10]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", encode_kwargs={"normalize_embeddings" : True})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="./chroma_db"
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline
from langchain_huggingface.llms import HuggingFacePipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=generator)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""Tu es un assistant chargé de répondre à des questions. Utilises les éléments de contexte récupérés ci-dessous pour répondre à la question. Si tu ne connais pas la réponse, dis simplement que tu ne la connais pas. Limites ta réponse à trois phrases maximum et restes concis.
Question: {question}
Context: {context}
Answer: """)

def rag_pipeline(query):
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=2
    )

    context = "\n\n".join(
        doc.page_content[:3000]
        for doc in retrieved_docs
    )

    prompt = prompt_template.invoke({
        "question": query,
        "context": context
    })

    response = llm.invoke(prompt)

    return response.strip(), retrieved_docs

In [ ]:
query = """
Les délibérations du tribunal arbitral sont-elles publiques ou secrètes ?
"""

response, retrieved_docs = rag_pipeline(query)

In [ ]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="
    border: 1px solid #ddd;
    border-radius: 12px;
    padding: 20px;
    margin: 15px 0;
    background-color: #f8f9fa;
    font-family: Arial, sans-serif;
">
    <h3 style="margin-top: 0;">🤖 Réponse de l'assistant</h3>

    <div style="
        background-color: white;
        border-radius: 8px;
        padding: 15px;
        margin-top: 20px;
        line-height: 1.6;
        border-left: 4px solid ;
    ">
        {response}
    </div>
</div>
"""))

In [ ]:
import re

def extraire_reference_texte(texte):
    correspondance = re.search(r"ARTICLE\s+(\d+)\s+([A-Z]{2,}?)(?=ARTICLE|\s|$)", texte.upper())
    if correspondance:
        return correspondance.group(2), correspondance.group(1)
    return None, None

def extraire_reference(document):
    return extraire_reference_texte(document.metadata["details"])

In [ ]:
def construire_lignes_soumission(question_id, retrieved_docs, reponse):
    document_reference, numero_article = extraire_reference(retrieved_docs[0])
    return [
        {"ID": f"{question_id}_Answer", "Target": reponse},
        {"ID": f"{question_id}_Document_de_Référence", "Target": document_reference},
        {"ID": f"{question_id}_Numéro_d'Article", "Target": numero_article},
    ]

In [ ]:
import os

if os.path.exists("Test.csv"):
    test = pd.read_csv("Test.csv")
    questions = dict(zip(test["ID"], test["question"]))
else:
    questions = {
        "Q1": "Les délibérations du tribunal arbitral sont-elles publiques ou secrètes ?"
    }

In [ ]:
lignes = []
for question_id, question in questions.items():
    reponse, retrieved_docs = rag_pipeline(question)
    lignes.extend(construire_lignes_soumission(question_id, retrieved_docs, reponse))

soumission = pd.DataFrame(lignes)
soumission.to_csv("submission.csv", index=False)
soumission

In [ ]:
def extraire_intitule(details):
    lignes = details.split("\n")
    if len(lignes) > 1 and lignes[0].strip() == "Analyses":
        return lignes[1].strip()
    return None

In [ ]:
lignes_test = []
for _, row in dataset.iterrows():
    document_reference, numero_article = extraire_reference_texte(row["details"])
    intitule = extraire_intitule(row["details"])
    if document_reference is None or intitule is None:
        continue
    lignes_test.append({
        "ID": f"Q{len(lignes_test) + 1}",
        "question": f"Quelle est la position de la jurisprudence OHADA sur : {intitule.lower()} ?",
        "reponse_reference": row["content"][:300],
        "document_reference_reference": document_reference,
        "numero_article_reference": numero_article,
    })
    if len(lignes_test) >= 20:
        break

jeu_test_synthetique = pd.DataFrame(lignes_test)
jeu_test_synthetique.to_csv("Test_synthetique.csv", index=False)
jeu_test_synthetique

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)

def calculer_metrique(reponse, document_reference, numero_article, reponse_reference, document_reference_reference, numero_article_reference):
    rouge1 = scorer.score(reponse_reference, reponse)["rouge1"].fmeasure
    exactitude_document = float(document_reference == document_reference_reference)
    exactitude_article = float(numero_article == numero_article_reference)
    return 0.5 * rouge1 + 0.25 * exactitude_document + 0.25 * exactitude_article

In [ ]:
scores = []
for _, row in jeu_test_synthetique.iterrows():
    reponse, retrieved_docs = rag_pipeline(row["question"])
    document_reference, numero_article = extraire_reference(retrieved_docs[0])
    score = calculer_metrique(
        reponse,
        document_reference,
        numero_article,
        row["reponse_reference"],
        row["document_reference_reference"],
        row["numero_article_reference"],
    )
    scores.append(score)

evaluation = jeu_test_synthetique.copy()
evaluation["score"] = scores
evaluation.to_csv("Evaluation_synthetique.csv", index=False)
evaluation["score"].mean()